In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import load_web_traffic
from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()



Project root: D:\MyML\datathon2026
Source path added: D:\MyML\datathon2026\src


In [2]:
from src.data import load_sales, load_web_traffic, load_promotions, load_order_items, load_orders
from src.features import add_time_features, add_lag_features
import pandas as pd

DATA_ROOT = project_root / "data/datathon-2026-round-1"

In [22]:
df_order_items = load_order_items(data_root=DATA_ROOT)
df_order = load_orders(data_root=DATA_ROOT)

# Attach order date to each order-item row, then count promo_id_2 usage per day.
order_items_with_date = df_order_items.merge(
    df_order[["order_id", "date"]], on="order_id", how="left"
)

promo2_by_date = (
    order_items_with_date.assign(
        promo2_used=lambda x: x["promo_id_2"].notna() & (x["promo_id_2"] != 0)
    )
    .groupby("date", as_index=False)["promo2_used"]
    .sum()
    .rename(columns={"promo2_used": "promo2_total_used"})
)

In [24]:
promo2_by_date.head()

,date,promo2_total_used
0,2012-07-04,0
1,2012-07-05,0
2,2012-07-06,0
3,2012-07-07,0
4,2012-07-08,0


In [25]:
from src.pipelines import train_validate_models, fit_models_full, predict_submission_from_models
from src.models import SklearnRegressorWrapper, SklearnRegressorConfig
df_sales = load_sales(data_root=DATA_ROOT)
df_traffic = load_web_traffic(data_root=DATA_ROOT)


df_traffic_test = pd.read_csv(f"{project_root}/results/webtraffic_predictions.csv")
X_traffic_test = df_traffic_test[["date", "sessions", "unique_visitors"]].drop(columns=["date"])

df = pd.merge(df_sales, df_traffic, on="date", how="left")
df = pd.merge(df, promo2_by_date, on="date", how="left")

df = add_time_features(df, date_col="date")

# returning_users = max(0, sessions - unique_visitors)
#df["returning_users"] = df["sessions"] - df["unique_visitors"]
#df["returning_users"] = df["returning_users"].clip(lower=0)

# returning_rate = (sessions - unique_visitors) / sessions
# df["returning_rate"] = (df["sessions"] - df["unique_visitors"]) / df["sessions"]

# df = add_lag_features(df, target_col="returning_rate", lags=[3], date_col="date")

In [ ]:
df.head()

,date,Revenue,COGS,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source,promo2_total_used,year,month,day,day_of_week,day_of_year,week_of_year,is_month_end,is_month_start,is_weekend
0,2013-01-01,5304546.99,4156070.20,9760,7253,39093,0.00514,102.9,organic_search,0,2013,1,1,1,1,1,0,1,0
1,2013-01-02,1606940.44,1237497.84,10456,8151,47611,0.00406,120.5,organic_search,0,2013,1,2,2,2,1,0,0,0
2,2013-01-03,2281680.01,1832133.02,10076,7458,36963,0.00401,263.6,direct,0,2013,1,3,3,3,1,0,0,0
3,2013-01-04,2376895.46,1940747.07,9973,8063,53078,0.00562,151.8,direct,0,2013,1,4,4,4,1,0,0,0
4,2013-01-05,2509462.77,1977027.71,10223,7882,36790,0.00525,168.6,referral,0,2013,1,5,5,5,1,0,0,1


In [27]:
df1 = df.drop(columns=['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source'])
df1 = df1[df1["date"] > "2019-01-01"]
targets = ["COGS", "Revenue"]
features = [col for col in df1.columns if col not in ["date", *targets]]
print(features)

model_config = SklearnRegressorConfig(
    model_type="xgboost", 
)
results = train_validate_models(
    df=df1,
    features=features,
    targets=targets,
    model_config=model_config,
)
results["metrics"]

['promo2_total_used', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'is_month_end', 'is_month_start', 'is_weekend']


{'COGS': {'mae': 571515.585145548,
  'rmse': 770756.556992878,
  'mape': 24.85197494714355,
  'smape': 20.940845106955518,
  'r2': 0.7367883770799903},
 'Revenue': {'mae': 676059.9579922945,
  'rmse': 944105.6913232538,
  'mape': 24.634797557205438,
  'smape': 22.565954736836613,
  'r2': 0.7015006378625642}}

In [ ]:
df_sales = load_sales(data_root=DATA_ROOT)
df_traffic = load_web_traffic(data_root=DATA_ROOT)

df_traffic_test = pd.read_csv(f"{project_root}/results/webtraffic_predictions.csv")
X_traffic_test = df_traffic_test[["date", "sessions", "unique_visitors"]].drop(columns=["date"])
df = pd.merge(df_sales, df_traffic, on="date", how="left")
df = add_time_features(df, date_col="date")
df.drop(columns=['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source'], inplace=True)
targets = ["COGS", "Revenue"]
features = [col for col in df.columns if col not in ["date", *targets]]
print(features)
model_config = SklearnRegressorConfig(
    model_type="xgboost", 
)
results = train_validate_models(
    df=df,
    features=features,
    targets=targets,
    model_config=model_config,
)
results["metrics"]

from src import models
from src.pipelines import predict_submission_from_models, fit_models_full
models = fit_models_full(
    df=df,
    features=features,
    targets=targets,
    model_config=model_config
)

submission = predict_submission_from_models(
    models=models,
    feature_cols=feature_cols,
    start_date="2023-01-01",
    end_date="2024-07-01",
    revenue_model_name="Revenue",
    ratio_model_name="ratio",
    lags=(1, 7, 14),
    windows=(7, 14),
    eps=1e-6,
)


['year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'is_month_end', 'is_month_start', 'is_weekend']


{'COGS': {'mae': 505952.48581566353,
  'rmse': 694086.1668286145,
  'mape': 20.086968510331392,
  'smape': 21.07987425184929,
  'r2': 0.7707160901867441},
 'Revenue': {'mae': 585314.4008703831,
  'rmse': 812629.873432484,
  'mape': 22.379391480814636,
  'smape': 21.461053364244695,
  'r2': 0.7621165636029108}}